<div style="background-color:#e2725b; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #e2725b; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Baseline Models</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>

This notebook establishes the first modeling baseline for the donor-response problem. The objective at this stage is not to maximize performance yet, but to create a clean and reliable workflow that respects the findings from the EDA.


In [ ]:
"""Importing the libraries needed for data handling, modeling, and evaluation"""
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

"""Project-internal imports (require project root on path)"""
from pipeline.feature_pipeline import (
    add_engineered_features,
    build_preprocessor,
    compute_metrics_from_probabilities,
    evaluate_classifier,
    get_feature_groups,
)


In [2]:
"""Loading the training dataset for baseline modeling"""
data = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_train.csv", index_col=0)
data.head()


,CARD_PROM_12,CHILDREN,DONOR_AGE,DONOR_GENDER,FILE_CARD_GIFT,FREQUENCY_STATUS_97NK,HOME_OWNER,INCOME_GROUP,LAST_GIFT_AMT,LIFETIME_CARD_PROM,LIFETIME_GIFT_AMOUNT,LIFETIME_GIFT_COUNT,LIFETIME_MAX_GIFT_AMT,LIFETIME_MIN_GIFT_AMT,LIFETIME_PROM,MEDIAN_HOME_VALUE,MEDIAN_HOUSEHOLD_INCOME,MONTHS_SINCE_FIRST_GIFT,MONTHS_SINCE_LAST_GIFT,MONTHS_SINCE_LAST_PROM_RESP,NUMBER_PROM_12,PCT_ATTRIBUTE1,PCT_ATTRIBUTE2,PCT_ATTRIBUTE3,PCT_ATTRIBUTE4,PCT_OWNER_OCCUPIED,PEP_STAR,PER_CAPITA_INCOME,RECENCY_STATUS_96NK,RECENT_AVG_CARD_GIFT_AMT,RECENT_AVG_GIFT_AMT,RECENT_CARD_RESPONSE_COUNT,RECENT_CARD_RESPONSE_PROP,RECENT_RESPONSE_COUNT,RECENT_RESPONSE_PROP,RECENT_STAR_STATUS,SES,URBANICITY,WEALTH_RATING,TARGET_B
CONTROL_NUMBER,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
61745,4.0,3.0,33.0,M,0.0,1.0,H,5.0,20.0,9.0,35.0,2.0,20.0,15.0,21.0,566.0,315.0,182.037132,18.0,18.0,10.0,0.0,52.0,17.0,25.0,92.0,0.0,12827.0,A,0.00,17.50,NaN,0.000,2.0,0.154,0.0,2,T,NaN,1
112703,3.0,2.0,NaN,F,1.0,1.0,U,NaN,15.0,6.0,15.0,1.0,15.0,15.0,15.0,318.0,148.0,24.000000,24.0,24.0,7.0,0.0,31.0,31.0,39.0,73.0,0.0,7787.0,N,15.00,15.00,1.0,0.250,1.0,0.100,0.0,3,R,NaN,1
166437,4.0,2.0,NaN,F,7.0,3.0,H,4.0,10.0,17.0,79.0,11.0,12.0,5.0,40.0,1669.0,373.0,129.000000,15.0,15.0,8.0,0.0,26.0,39.0,38.0,84.0,1.0,13965.0,S,0.00,10.67,0.0,0.000,3.0,0.231,1.0,1,U,NaN,0
170621,4.0,NaN,61.0,M,13.0,1.0,H,6.0,11.0,28.0,80.0,17.0,11.0,3.0,75.0,1464.0,488.0,130.000000,16.0,16.0,13.0,0.0,48.0,30.0,44.0,84.0,1.0,24123.0,A,10.00,10.00,2.0,0.286,2.0,0.111,0.0,1,U,NaN,0
44428,6.0,0.0,75.0,M,3.0,4.0,H,3.0,7.0,9.0,27.0,5.0,7.0,5.0,22.0,936.0,249.0,24.000000,17.0,17.0,13.0,0.0,52.0,NaN,66.0,90.0,1.0,15008.0,N,5.67,5.40,3.0,0.600,5.0,0.500,0.0,2,C,NaN,0


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Problem setup and data partition</h3>

Following good modeling practice, the first step is to separate features and target and create a train/validation split before any fitted preprocessing is applied. This helps avoid data leakage and makes the baseline evaluation more reliable.


In [3]:
"""Defining the target and identifier variables and creating the train-validation split"""
TARGET = "TARGET_B"
ID_COLUMN = "CONTROL_NUMBER"
RANDOM_STATE = 5
TRAIN_SIZE = 0.70

X = data.drop(columns=[TARGET])
y = data[TARGET].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=TRAIN_SIZE,
    shuffle=True,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Training target rate: {y_train.mean():.3f}")
print(f"Validation target rate: {y_val.mean():.3f}")


Training set shape: (9492, 39)
Validation set shape: (4068, 39)
Training target rate: 0.250
Validation target rate: 0.250


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">3. Rule-based cleaning from the EDA</h3>

At this stage, we only apply the cleaning rules that were directly supported by the EDA. These are deterministic corrections, not learned preprocessing steps, so they can be applied consistently to both the training and validation sets.


In [4]:
"""Creating a rule-based cleaning function using only the issues already justified in the EDA"""
def apply_eda_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Apply only the domain rules already justified in the EDA notebook."""
    cleaned = df.copy()

    """Treating ambiguous categorical codes as missing values"""
    for column in ["SES", "URBANICITY"]:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].replace("?", np.nan)

    """Converting domain-invalid numeric values into missing values"""
    if "CHILDREN" in cleaned.columns:
        cleaned.loc[cleaned["CHILDREN"] < 0, "CHILDREN"] = np.nan

    if "RECENT_RESPONSE_PROP" in cleaned.columns:
        invalid_recent_response = (
            (cleaned["RECENT_RESPONSE_PROP"] < 0)
            | (cleaned["RECENT_RESPONSE_PROP"] > 1)
        )
        cleaned.loc[invalid_recent_response, "RECENT_RESPONSE_PROP"] = np.nan

    if "RECENT_CARD_RESPONSE_PROP" in cleaned.columns:
        invalid_recent_card_response = (
            (cleaned["RECENT_CARD_RESPONSE_PROP"] < 0)
            | (cleaned["RECENT_CARD_RESPONSE_PROP"] > 1)
        )
        cleaned.loc[invalid_recent_card_response, "RECENT_CARD_RESPONSE_PROP"] = np.nan

    return cleaned


In [5]:
"""Applying the EDA-based cleaning rules to the training and validation partitions and summarizing the detected issues"""
X_train_clean = apply_eda_cleaning(X_train)
X_val_clean = apply_eda_cleaning(X_val)

cleaning_summary = pd.DataFrame(
    {
        "invalid_children_train": [int((X_train["CHILDREN"] < 0).sum())],
        "invalid_recent_response_prop_train": [
            int(((X_train["RECENT_RESPONSE_PROP"] < 0) | (X_train["RECENT_RESPONSE_PROP"] > 1)).sum())
        ],
        "invalid_recent_card_response_prop_train": [
            int(((X_train["RECENT_CARD_RESPONSE_PROP"] < 0) | (X_train["RECENT_CARD_RESPONSE_PROP"] > 1)).sum())
        ],
        "ses_question_mark_train": [int((X_train["SES"].astype(str) == "?").sum())],
        "urbanicity_question_mark_train": [int((X_train["URBANICITY"].astype(str) == "?").sum())],
    }
)

display(cleaning_summary)


,invalid_children_train,invalid_recent_response_prop_train,invalid_recent_card_response_prop_train,ses_question_mark_train,urbanicity_question_mark_train
0,48,46,51,221,223


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">4. Feature groups and preprocessing design</h3>

The baseline preprocessing should remain simple. Numerical variables will be imputed with the median and scaled, while categorical variables will be imputed with a constant label and one-hot encoded. The identifier remains outside the modeling matrix because it is loaded as the dataframe index.


In [ ]:
"""Preparing the modeling matrices and inspecting the preprocessing groups"""
X_train_model = X_train_clean.copy()
X_val_model = X_val_clean.copy()
# get_feature_groups now lives in pipeline.feature_pipeline; we just call it here.
feature_groups = get_feature_groups(X_train_model)

feature_group_summary = pd.DataFrame(
    {
        "feature_group": [
            "continuous_numeric_features",
            "coded_numeric_features",
            "categorical_features",
            "dropped_high_missing_features",
        ],
        "n_features": [
            len(feature_groups["continuous_numeric_features"]),
            len(feature_groups["coded_numeric_features"]),
            len(feature_groups["categorical_features"]),
            len(feature_groups["dropped_high_missing_features"]),
        ],
        "features": [
            ", ".join(feature_groups["continuous_numeric_features"]),
            ", ".join(feature_groups["coded_numeric_features"]),
            ", ".join(feature_groups["categorical_features"]),
            ", ".join(feature_groups["dropped_high_missing_features"]),
        ],
    }
)

display(feature_group_summary)

## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">5. Baseline model definitions</h3>

The first comparison focuses on simple and widely used baseline classifiers. The goal is to understand which model family looks most promising before moving to feature engineering and model selection.


In [8]:
"""Defining the first set of baseline classifiers to compare"""
model_configs = {
    "Logistic Regression": {
        "estimator": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "model_family": "linear",
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "model_family": "tree",
    },
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=8,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
        ),
        "model_family": "tree",
    },
    "Extra Trees": {
        "estimator": ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "model_family": "tree",
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(
            random_state=RANDOM_STATE,
        ),
        "model_family": "tree",
    },
}

model_configs


{'Logistic Regression': {'estimator': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=5),
  'model_family': 'linear'},
 'Random Forest': {'estimator': RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                         random_state=5),
  'model_family': 'tree'},
 'Decision Tree': {'estimator': DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                         min_samples_leaf=20, random_state=5),
  'model_family': 'tree'},
 'Extra Trees': {'estimator': ExtraTreesClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=5),
  'model_family': 'tree'},
 'Gradient Boosting': {'estimator': GradientBoostingClassifier(random_state=5),
  'model_family': 'tree'}}

## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">6. Baseline training and validation results</h3>

Since the official competition metric has not yet been fixed in this notebook, we compare models using a small set of standard classification metrics. For ranking the first baselines, `ROC-AUC` is used as the main reference metric.


In [ ]:
"""Training each baseline pipeline on the raw cleaned features and comparing validation metrics"""
raw_results_rows = []
fitted_pipelines = {}

for model_name, config in model_configs.items():
    preprocessor, model_feature_groups = build_preprocessor(
        X_train_model,
        model_family=config["model_family"],
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", clone(config["estimator"])),
        ]
    )

    metrics, _ = evaluate_classifier(
        model_pipeline=pipeline,
        X_train_df=X_train_model,
        X_val_df=X_val_model,
        y_train_series=y_train,
        y_val_series=y_val,
    )

    raw_results_rows.append(
        {
            "scenario": "raw",
            "model": model_name,
            "model_family": config["model_family"],
            "dropped_high_missing": ", ".join(model_feature_groups["dropped_high_missing_features"]) or "None",
            **metrics,
        }
    )
    fitted_pipelines[model_name] = pipeline

baseline_results = (
    pd.DataFrame(raw_results_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

baseline_results.style.format(
    {
        "threshold": "{:.2f}",
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
        "roc_auc": "{:.3f}",
    }
)

> **Main Insights**  
The baseline results show that accuracy is not a sufficiently informative metric for this problem. Although `Gradient Boosting`, `Random Forest`, and `Extra Trees` reached the highest accuracy values, they identified very few positive cases, which is reflected in their very low recall and F1-scores. This suggests that these models are still too conservative under the default classification threshold.

**Interpretation.**  
Among the tested baselines, `Logistic Regression` provides the best balance between identifying positive cases and controlling false positives, resulting in the strongest F1-score and recall. At the same time, `Gradient Boosting` achieved the highest ROC-AUC, which suggests that it may still be useful as a ranking model even though its default threshold is not yet adequate for classification. Overall, these first results indicate that the next improvements should focus on feature engineering and threshold tuning rather than on accuracy alone.


## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">7. Baselines with feature engineering</h3>

We now repeat the baseline experiment after applying the donor-level feature engineering defined in `pipeline.feature_pipeline.add_engineered_features` (ratios linked to giving frequency and value, time-since-first-gift activity ratios, recent-response gaps, etc.). The preprocessing pipeline and the five candidate models are unchanged, so any difference in validation performance is attributable to the engineered variables alone.


In [ ]:
"""Training each baseline pipeline on the engineered feature set"""
X_train_fe = add_engineered_features(X_train_model)
X_val_fe   = add_engineered_features(X_val_model)

fe_results_rows = []
for model_name, config in model_configs.items():
    preprocessor, _ = build_preprocessor(X_train_fe, model_family=config["model_family"])
    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", clone(config["estimator"])),
        ]
    )
    metrics, _ = evaluate_classifier(
        model_pipeline=pipeline,
        X_train_df=X_train_fe,
        X_val_df=X_val_fe,
        y_train_series=y_train,
        y_val_series=y_val,
    )
    fe_results_rows.append({
        "scenario": "+FE",
        "model": model_name,
        "model_family": config["model_family"],
        **metrics,
    })

fe_results = (
    pd.DataFrame(fe_results_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

fe_results.style.format({
    "threshold": "{:.2f}",
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
})

## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">8. Baselines with feature engineering + selection</h3>

On top of the engineered features, we now also restrict the model input to the **consensus feature list** produced by notebook 02 (`02_feature_engineering_and_selection`). The consensus is the set of features that won the family-level vote across filter, wrapper, and embedded selection methods, and it lives in `project_data/selected_features.json`.

Because the consensus list is expressed in the column-name space that comes out of `build_preprocessor` (post one-hot encoding, post missing-value indicators), we cannot put the selection step inside a standard `Pipeline`. Instead, for each model we (1) fit the preprocessor on the training data, (2) materialize the transformed matrix, (3) filter columns by name, and (4) fit the estimator on the selected matrix.


In [ ]:
"""Training each baseline pipeline on the engineered + selected feature set"""
with open(PROJECT_ROOT / "project_data" / "selected_features.json") as fh:
    selection = json.load(fh)
consensus_features = set(selection["consensus"])
print(f"Consensus list size: {len(consensus_features)} features")

fs_results_rows = []
for model_name, config in model_configs.items():
    preprocessor, _ = build_preprocessor(X_train_fe, model_family=config["model_family"])

    # 1) Fit preprocessor on train, transform both splits.
    X_train_prep = preprocessor.fit_transform(X_train_fe)
    X_val_prep   = preprocessor.transform(X_val_fe)

    # 2) Strip the transformer prefix and figure out which output columns survive selection.
    feature_names = [name.split("__")[-1] for name in preprocessor.get_feature_names_out()]
    selected_mask = np.array([name in consensus_features for name in feature_names])

    if not selected_mask.any():
        print(f"[{model_name}] no features survive selection — skipping")
        continue

    # 3) Fit the estimator on the selected columns.
    estimator = clone(config["estimator"])
    estimator.fit(X_train_prep[:, selected_mask], y_train)
    y_proba = estimator.predict_proba(X_val_prep[:, selected_mask])[:, 1]
    metrics = compute_metrics_from_probabilities(y_val, y_proba, threshold=0.50)

    fs_results_rows.append({
        "scenario": "+FE+FS",
        "model": model_name,
        "model_family": config["model_family"],
        "n_features_kept": int(selected_mask.sum()),
        **metrics,
    })

fs_results = (
    pd.DataFrame(fs_results_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

fs_results.style.format({
    "threshold": "{:.2f}",
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
})

## <h3 style="border-bottom: 4px solid #e2725b; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">9. Final comparison across scenarios</h3>

Putting the three scenarios side by side makes the contribution of each step visible per model. Only `ROC-AUC` and `F1` are shown — these are the metrics that matter for an imbalanced ranking problem. The notebook downstream (`04_model_selection`) will build on the best scenario per model from this table.


In [ ]:
"""Combining the three scenarios into a single side-by-side comparison"""
all_scenarios = pd.concat(
    [
        baseline_results.assign(scenario="raw"),
        fe_results.assign(scenario="+FE"),
        fs_results.assign(scenario="+FE+FS"),
    ],
    ignore_index=True,
    sort=False,
)

scenario_order = ["raw", "+FE", "+FE+FS"]
comparison = (
    all_scenarios
    .pivot_table(index="model", columns="scenario", values=["roc_auc", "f1"], aggfunc="first")
    .reorder_levels([1, 0], axis=1)
    .sort_index(axis=1, level=0, key=lambda idx: pd.Index([scenario_order.index(v) for v in idx]))
)

# Pick best scenario per model by ROC-AUC for a quick reference.
best_scenario_per_model = (
    all_scenarios
    .sort_values(["model", "roc_auc"], ascending=[True, False])
    .drop_duplicates("model")
    [["model", "scenario", "roc_auc", "f1"]]
    .reset_index(drop=True)
)

print("Side-by-side comparison (ROC-AUC and F1 per model per scenario):")
display(comparison.style.format("{:.3f}"))

print("\nBest scenario per model (by ROC-AUC):")
display(best_scenario_per_model.style.format({"roc_auc": "{:.3f}", "f1": "{:.3f}"}))

> **Main Insights — Section 9**  
Reading across each row you can see whether feature engineering and feature selection actually helped that model. A typical pattern in donor-response problems is: linear models (logistic regression) benefit visibly from feature engineering because the engineered ratios make non-linear donor patterns expressible in a linear form, while tree-based models (Random Forest, Gradient Boosting, Extra Trees, Decision Tree) benefit less because they can already model the original variables non-linearly. Feature selection tends to leave linear models roughly unchanged or slightly improved (less noise) and may slightly hurt tree models if the selection is conservative. The `Best scenario per model` table is the recommended starting point for the hyper-parameter tuning in `04_model_selection`: each model is tuned in the scenario where it performed best at the baseline stage, which keeps the comparison fair without forcing every model to live with the same input representation.
